# Bhagavad Gita Knowledge Graph — Loader

Loads chapters, verses, speakers, addressees, epithets, setting, and lemmatized terms
from the English translations into the local Neo4j `TheGitaProject` database.

- **Spec:** `docs/superpowers/specs/2026-08-20-gita-knowledge-graph-design.md`
- Deterministic + idempotent: re-running rebuilds the graph with no duplicates.
- Requires a local Neo4j with a `TheGitaProject` database and a `gita-knowledge-graph/.env`
  (copy from `.env.example`).

In [1]:
import collections
import sys
from pathlib import Path

import spacy
from dotenv import load_dotenv
from neo4j import GraphDatabase


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))  # make gita_kg importable regardless of cwd

import gita_kg as gk

## 1. Config & connect

In [7]:
load_dotenv(PKG / ".env", override=True)  # .env is the source of truth
cfg = gk.load_config()
driver = GraphDatabase.driver(cfg.uri, auth=(cfg.user, cfg.password))
driver.verify_connectivity()
print("connected:", cfg.uri, "->", cfg.database)

connected: bolt://localhost:7687 -> neo4j


## 2. spaCy pipeline (EntityRuler for epithets)

In [3]:
nlp = gk.build_epithet_ruler(spacy.load("en_core_web_sm"))

## 3. Parse the verses

In [4]:
VERSES_DIR = ROOT / "data/TheGitaProject/Verses"
records = gk.build_records(VERSES_DIR, nlp)
print(f"parsed {len(records)} verses")
print("speakers:", collections.Counter(r.speaker for r in records))

parsed 701 verses
speakers: Counter({'Krishna': 574, 'Arjuna': 86, 'Sanjaya': 40, 'Dhritarashtra': 1})


## 4. Load into Neo4j (constraints → seeds → verses)

In [12]:
def run_ops(ops):
    with driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)


run_ops(gk.constraint_ops())
run_ops(gk.seed_ops())
run_ops(gk.verse_ops(records))
print("load complete")

load complete


## 5. Verification

In [13]:
def one(cypher):
    with driver.session(database=cfg.database) as s:
        return s.run(cypher).single()[0]


print("verses:", one("MATCH (v:Verse) RETURN count(v)"))
print(
    "no SPOKEN_BY:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:SPOKEN_BY]->() RETURN count(v)"),
)
print(
    "no ADDRESSED_TO:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:ADDRESSED_TO]->() RETURN count(v)"),
)
print("terms:", one("MATCH (t:Term) RETURN count(t)"))
print("epithet edges:", one("MATCH ()-[r:USES_EPITHET]->() RETURN count(r)"))

verses: 701
no SPOKEN_BY: 0
no ADDRESSED_TO: 0
terms: 1171
epithet edges: 32


In [10]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (v:Verse)-[:SPOKEN_BY]->(:Person {name:'Arjuna'}) "
        "RETURN v.id AS id ORDER BY v.chapter, v.verse LIMIT 10"
    ).values()
print("Arjuna's first verses:", rows)

Arjuna's first verses: [['1.21'], ['1.22'], ['1.23'], ['1.28'], ['1.29'], ['1.30'], ['1.31'], ['1.32'], ['1.33'], ['1.34']]


In [11]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (:Chapter {number:2})-[:HAS_VERSE]->(v)-[m:MENTIONS_TERM]->(t) "
        "RETURN t.lemma AS term, sum(m.count) AS n ORDER BY n DESC LIMIT 10"
    ).values()
print("Chapter 2 top terms:", rows)

Chapter 2 top terms: [['wisdom', 13], ['say', 13], ['sens', 10], ['pleasure', 9], ['grieve', 9], ['attain', 9], ['speak', 9], ['desire', 9], ['man', 9], ['word', 8]]


## 6. Close

In [14]:
driver.close()